In [ ]:
# https://learnopencv.com/fine-tuning-bert/

In [1]:
from pathlib import Path

import torch

from collections.abc import Callable

from datasets import Dataset, load_dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

import evaluate
import glob
import numpy as np
import pandas as pd

from pathlib import Path


import sys

sys.path.insert(0, str(Path.cwd().parent))


from finetuning.commons import PipelineData, prepare_data, parse_pubtator, build_training_samples, samples_to_rels_like_df

/nix/store/vcapnvnswfafrsqa88vi4xip12wfghnd-python3-3.12.10-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
OUT_DIR = 'relations-bert'

# NOTE: these are the defaults might change according to avaibale VRAM
# -> BATCH SIZE and LR are halved if less than 8GB of VRAM is detected
BATCH_SIZE = 32
NUM_PROCS = 32
LR = 0.00005
EPOCHS = 5
MODEL = 'NeuML/pubmedbert-base-embeddings'
CACHE_DIR = Path("cache")

PUBTATOR_FILE = Path.home() / "git/LLMs_for_NEL/Data/BioRED/Dev.PubTator"



def load_or_cache(split: str, prefix: str, build: Callable[[], Dataset], use_cache: bool = True) -> Dataset:
    """Load a Hugging Face ``Dataset`` from disk cache or build and persist it.

    Cached data is stored under ``CACHE_DIR / f"{prefix}_{split}"`` (default: ``cache/``).
    On a cache hit, ``load_from_disk`` is used and ``build`` is not called.
    On a miss, ``build()`` runs once, the result is saved with ``save_to_disk``, then returned.

    Args:
        split: Split identifier used in the cache directory name (e.g. ``"train"``, ``"validation"``).
        prefix: Stage prefix distinguishing pipeline steps (e.g. ``"raw"``, ``"tokenized"``).
        build: Zero-argument callable that produces the dataset when the cache is missing.

    Returns:
        The dataset for the given split, either loaded from cache or freshly built.

    Examples:
        Download a Hub split and cache it as ``cache/raw_train/``::

            train = load_or_cache(
                "train",
                "raw",
                lambda: load_dataset("ccdv/arxiv-classification", split="train"),
            )

        Tokenize an in-memory split and cache as ``cache/tokenized_train/``::

            tokenized_train = load_or_cache(
                "train",
                "tokenized",
                lambda: train.map(preprocess_function, batched=True, batch_size=32),
            )

    Note:
        Delete the matching folder under ``cache/`` to force a rebuild after changing
        ``build``, the source data, or preprocessing.
    """
    cache_path = CACHE_DIR / f"{prefix}_{split}"
    if cache_path.exists() and use_cache:
        print(f"Loading {prefix} {split} from cache/")
        return load_from_disk(cache_path)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    dataset = build()
    dataset.save_to_disk(cache_path)
    return dataset


In [ ]:
MODE: str = "cpu"
if torch.backends.mps.is_available():
    MODE = "mps"
elif torch.cuda.is_available():
    MODE = "cuda"
else:
    print("No GPU or MPS available - uising CPU")

print(f"Using {MODE} for training")


hardware_specific_args = {}

if MODE == "mps":
    hardware_specific_args["fp16"] = False
    hardware_specific_args["dataloader_num_workers"] = 0
    hardware_specific_args["per_device_train_batch_size"] = BATCH_SIZE
    hardware_specific_args["per_device_eval_batch_size"] = BATCH_SIZE
    hardware_specific_args["learning_rate"] = LR
    

elif MODE == "cuda":
    # Check total GPU VRAM 
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA device VRAM: {total_vram_gb:.2f} GB")

    # Default: assume >8GB VRAM
    batch_div = 1
    lr_div = 1

    # 2070super only has 8gigs of VRAM :')
    if total_vram_gb <= 8.5:
        print("Detected ~8GB of VRAM or less, reducing batch size and learning rate.")
        batch_div = 2
        lr_div = 2

    hardware_specific_args["fp16"] = True
    hardware_specific_args["per_device_train_batch_size"] = BATCH_SIZE // batch_div
    hardware_specific_args["per_device_eval_batch_size"] = BATCH_SIZE // batch_div
    hardware_specific_args["learning_rate"] = LR / lr_div

    print(f"Using {hardware_specific_args['per_device_train_batch_size']} for training")
    print(f"Using {hardware_specific_args['per_device_eval_batch_size']} for evaluation")
    print(f"Using {hardware_specific_args['learning_rate']} for learning rate")



In [ ]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=EPOCHS,
    report_to='tensorboard',
    **hardware_specific_args,
)

In [ ]:


meta_df, anns_df, rels_df = parse_pubtator(PUBTATOR_FILE)

samples = build_training_samples(meta_df, anns_df, rels_df)


samples = samples_to_rels_like_df(samples)
samples["prompt"] = samples.apply(
    lambda row: f"{row['entity_a_text']} -> {row['relation_type']} -> {row['entity_b_text']} \n{row['abstract']}",
    axis=1
)



from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(samples, test_size=0.1, random_state=42, stratify=None)
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
# print(train_dataset)
# print(valid_dataset)
# print(test_dataset)

In [ ]:
train_dataset[0]

In [ ]:
train_dataset[0].keys()

In [ ]:
label2id: dict[str, int] = {"false": 0, "true": 1}
id2label: dict[int, str] = {0: "false", 1: "true"}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
def preprocess_function(examples, key: str = "prompt"):
    return tokenizer(
        examples[key],
        truncation=True,
        padding=True,
        max_length=512,
    )


In [ ]:
def _tokenize(dataset: Dataset) -> Dataset:
    return dataset.map(
        preprocess_function,
        batched=True,
        batch_size=BATCH_SIZE,
        num_proc=NUM_PROCS,
    )


tokenized_train = load_or_cache("train", "bio_red_tokenized", lambda: _tokenize(train_dataset), use_cache=True)
tokenized_valid = load_or_cache("valid", "bio_red_tokenized", lambda: _tokenize(valid_dataset), use_cache=True)
# tokenized_test = load_or_cache("test", "bio_red_tokenized", lambda: _tokenize(test_dataset))

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
tokenized_sample = preprocess_function(train_dataset[0])
print(tokenized_sample)
print(f"Length of tokenized IDs: {len(tokenized_sample.input_ids)}")
print(f"Length of attention mask: {len(tokenized_sample.attention_mask)}")

In [ ]:
tokenized_sample = preprocess_function(train_dataset[0])
print(tokenized_sample)

In [ ]:
accuracy = evaluate.load('precision')
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
if MODE != "cpu":
    model = model.to(MODE)
print(f"Moving model to {MODE}")


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
history = trainer.train()

In [ ]:
model.save_pretrained(f"26_05_2026_pubmedbert_bio_red-relations")

In [ ]:
	
trainer.evaluate(tokenized_valid)

In [ ]:
CHECKPOINT_DIR = Path(OUT_DIR) / "checkpoint-566"

model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

if MODE != "cpu":
    model = model.to(MODE)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
AutoModelForSequenceClassification.from_pretrained(f"arxiv_bert/checkpoint-3550")
 
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
classify = pipeline(task='text-classification', model=model, tokenizer=tokenizer)
 
all_files = glob.glob('arxiv_custom_inference_data/*')
for file_name in all_files:
    file = open(file_name)
    content = file.read()
    print(content)
    result = classify(content)
    print('PRED: ', result)
    print('GT: ', file_name.split('_')[-1].split('.txt')[0])
    print('\n')